In [2]:
import pandas as pd

df = pd.read_csv("/content/merged_clean_ipl.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")


/tmp/ipython-input-1538564286.py:3: DtypeWarning: Columns (43,51,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/merged_clean_ipl.csv")


In [3]:
batting_match = df.groupby(
    ["match_id", "date", "batter", "batting_team", "bowling_team", "venue"]
).agg(
    runs=("runs_batter", "sum"),
    balls=("valid_ball", "sum"),
    fours=("runs_batter", lambda x: (x == 4).sum()),
    sixes=("runs_batter", lambda x: (x == 6).sum())
).reset_index()

bowling_match = df.groupby(
    ["match_id", "date", "bowler", "bowling_team", "batting_team", "venue"]
).agg(
    wickets=("bowler_wicket", "sum"),
    runs_conceded=("runs_bowler", "sum"),
    balls_bowled=("valid_ball", "sum")
).reset_index()

bowling_match["overs"] = bowling_match["balls_bowled"] / 6


In [13]:
batting_match.head()
bowling_match.head()


,match_id,date,bowler,bowling_team,batting_team,venue,wickets,runs_conceded,balls_bowled,overs,avg_wkts_last_5,venue_avg_wkts
0,335982,2008-04-18,AA Noffke,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1.0,40,24,4.0,1.0,1.000000
1,335982,2008-04-18,AB Agarkar,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,3.0,25,24,4.0,3.0,1.500000
2,335982,2008-04-18,AB Dinda,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,2.0,9,18,3.0,2.0,1.666667
3,335982,2008-04-18,CL White,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0.0,24,6,1.0,0.0,0.500000
4,335982,2008-04-18,I Sharma,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,1.0,7,18,3.0,1.0,0.666667


In [5]:
batting_match = batting_match.sort_values("date")

batting_match["avg_runs_last_5"] = (
    batting_match.groupby("batter")["runs"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

bowling_match["avg_wkts_last_5"] = (
    bowling_match.groupby("bowler")["wickets"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)



In [14]:
batting_match[batting_match["batter"] == "V Kohli"][
    ["date", "runs", "avg_runs_last_5"]
].head(8)


,date,runs,avg_runs_last_5
13,2008-04-18,1,1.000000
56,2008-04-20,23,12.000000
168,2008-04-26,13,12.333333
224,2008-04-28,12,12.250000
252,2008-04-30,1,10.000000
781,2008-05-03,38,17.400000
381,2008-05-05,34,19.600000
531,2008-05-12,21,21.200000


In [11]:
batting_match["venue_avg_runs"] = (
    batting_match.groupby(["batter", "venue"])["runs"]
    .transform("mean")
)

bowling_match["venue_avg_wkts"] = (
    bowling_match.groupby(["bowler", "venue"])["wickets"]
    .transform("mean")
)



In [12]:
batting_match[
    ["batter", "venue", "runs", "venue_avg_runs"]
].head()





,batter,venue,runs,venue_avg_runs
0,AA Noffke,M Chinnaswamy Stadium,9,9.000000
15,Z Khan,M Chinnaswamy Stadium,3,5.500000
14,W Jaffer,M Chinnaswamy Stadium,6,25.000000
12,SC Ganguly,M Chinnaswamy Stadium,10,16.333333
11,SB Joshi,M Chinnaswamy Stadium,3,3.000000


In [15]:
batting_match["avg_runs_vs_team"] = (
    batting_match.groupby(["batter", "bowling_team"])["runs"]
    .transform("mean")
)

bowling_match["avg_wkts_vs_team"] = (
    bowling_match.groupby(["bowler", "batting_team"])["wickets"]
    .transform("mean")
)


In [17]:
print(batting_match[["batter", "bowling_team", "runs", "avg_runs_vs_team"]].head())
print(bowling_match[["bowler", "batting_team", "wickets", "avg_wkts_vs_team"]].head())


        batter                 bowling_team  runs  avg_runs_vs_team
0    AA Noffke        Kolkata Knight Riders     9          9.000000
15      Z Khan        Kolkata Knight Riders     3          5.333333
14    W Jaffer        Kolkata Knight Riders     6          6.000000
12  SC Ganguly  Royal Challengers Bangalore    10         13.857143
11    SB Joshi        Kolkata Knight Riders     3          3.000000
       bowler                 batting_team  wickets  avg_wkts_vs_team
0   AA Noffke        Kolkata Knight Riders      1.0               1.0
1  AB Agarkar  Royal Challengers Bangalore      3.0               1.0
2    AB Dinda  Royal Challengers Bangalore      2.0               1.2
3    CL White        Kolkata Knight Riders      0.0               0.0
4    I Sharma  Royal Challengers Bangalore      1.0               1.0


In [18]:
batting_match["career_avg_runs"] = (
    batting_match.groupby("batter")["runs"].transform("mean")
)

bowling_match["career_avg_wkts"] = (
    bowling_match.groupby("bowler")["wickets"].transform("mean")
)


In [19]:
print(batting_match[["batter", "runs", "career_avg_runs"]].head())
print(bowling_match[["bowler", "wickets", "career_avg_wkts"]].head())


        batter  runs  career_avg_runs
0    AA Noffke     9         9.000000
15      Z Khan     3         5.210526
14    W Jaffer     6        16.250000
12  SC Ganguly    10        24.089286
11    SB Joshi     3         3.000000
       bowler  wickets  career_avg_wkts
0   AA Noffke      1.0         1.000000
1  AB Agarkar      3.0         0.725000
2    AB Dinda      2.0         0.891304
3    CL White      0.0         0.166667
4    I Sharma      1.0         0.900000
